In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01"
)

In [3]:
from pathlib import Path

# Inspect the actual Kaggle input directories
for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and p.name == "bank-full.csv":
        print("Correct file path:", p)

Correct file path: /kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-full.csv


In [4]:

df1 = pd.read_csv(DATA_DIR / "bank-full.csv", sep=";")
df2 = pd.read_csv(DATA_DIR / "bank.csv", sep=";")
df3 = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=";")
df4 = pd.read_csv(DATA_DIR / "bank-additional.csv", sep=";")

# PIPELINE

Logistic Regression: OneHotEncoder + StandardScaler.

Decision Tree, Random Forest, XGBoost: OneHotEncoder + numerical passthrough.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

1) df2 and df4 are smaller data sets selected randomly from df1 and df3. For now we will train baseline logistic regerssion, decision treee, random forest and xgboost models on our larger datasets.
2) out of time analysis will bw handled later separately


In [6]:
df1_scores={}
df3_scores={}

In [7]:
# Separate features and target
X1 = df1.drop(columns=['y'])
y1 = df1['y'].map({'no': 0, 'yes': 1})

# Train-test split (80:20)
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1,
    test_size=0.2,
    stratify=y1,
    random_state=42
)
print(X1_train.shape, X1_test.shape, y1_train.shape, y1_test.shape)

(36168, 16) (9043, 16) (36168,) (9043,)


In [8]:

num1_cols = X1_train.select_dtypes(include=np.number).columns.tolist()
cat1_cols = X1_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features in df1:", num1_cols)
print("Categorical features in df1:", cat1_cols)


Numerical features in df1: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical features in df1: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [9]:

# Numerical preprocessing
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both preprocessors
preprocessor1= ColumnTransformer([
    ('num', num_pipeline, num1_cols),
    ('cat', cat_pipeline, cat1_cols)
])


In [10]:
# separate pipeline for lr due to its need of nuemeric feature scaling
lr_pipeline1 = Pipeline([
    ('preprocessor', preprocessor1),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train
lr_pipeline1.fit(X1_train, y1_train)

# Predictions
y1_pred = lr_pipeline1.predict(X1_test)
y1_prob = lr_pipeline1.predict_proba(X1_test)[:, 1]

In [11]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score

)
from sklearn.metrics import recall_score
# print("Confusion Matrix:\n", confusion_matrix(y1_test, y1_pred))
# print("\nClassification Report:\n", classification_report(y1_test, y1_pred))
# print("ROC-AUC:", roc_auc_score(y1_test, y1_prob))
# print("Average Precision (PR-AUC):", average_precision_score(y1_test, y1_prob))



print(f"\n{'='*50}")
print("Logistic Regression on df1")
print(f"{'='*50}")

print("\nConfusion Matrix:")
print(confusion_matrix(y1_test, y1_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y1_test, y1_pred, zero_division=0))

print("Recall:", recall_score(y1_test, y1_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y1_test, y1_prob))
print("Average Precision:", average_precision_score(y1_test, y1_prob))
df1_scores["Logistic Regression"]=[recall_score(y1_test, y1_pred, zero_division=0), roc_auc_score(y1_test, y1_prob), average_precision_score(y1_test, y1_prob)]


Logistic Regression on df1

Confusion Matrix:
[[7782  203]
 [ 690  368]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      7985
           1       0.64      0.35      0.45      1058

    accuracy                           0.90      9043
   macro avg       0.78      0.66      0.70      9043
weighted avg       0.89      0.90      0.89      9043

Recall: 0.34782608695652173
ROC-AUC: 0.9055740146044154
Average Precision: 0.5451315654132785


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Preprocessing: One-Hot Encoding only
preprocessor1 = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat1_cols)
    ],
    remainder='passthrough'
)


In [13]:

# Define models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    )
}


In [14]:
# Train and evaluate models
fitted_models1 = {}

for name, model in models.items():

    clf1 = Pipeline([
        ('preprocessor', preprocessor1),
        ('classifier', model)
    ])

    # Train
    clf1.fit(X1_train, y1_train)

    # Predictions
    y_pred = clf1.predict(X1_test)
    y_prob = clf1.predict_proba(X1_test)[:, 1]

    # Evaluation
    print(f"\n{'='*50}")
    print(f"Model: {name}", "on df1")
    print(f"{'='*50}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y1_test, y_pred, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(y1_test, y_pred, zero_division=0))

    print("Recall:", recall_score(y1_test, y_pred, zero_division=0))
    print("ROC-AUC:", roc_auc_score(y1_test, y_prob))
    print("Average Precision:", average_precision_score(y1_test, y_prob))
    df1_scores[name]=[recall_score(y1_test, y_pred, zero_division=0), roc_auc_score(y1_test, y_prob), average_precision_score(y1_test, y_prob)]    
    fitted_models1[name] = clf1



Model: Decision Tree on df1

Confusion Matrix:
[[7403  582]
 [ 560  498]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      7985
           1       0.46      0.47      0.47      1058

    accuracy                           0.87      9043
   macro avg       0.70      0.70      0.70      9043
weighted avg       0.87      0.87      0.87      9043

Recall: 0.4706994328922495
ROC-AUC: 0.6989063852000384
Average Precision: 0.27897109037469303

Model: Random Forest on df1

Confusion Matrix:
[[7774  211]
 [ 630  428]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      7985
           1       0.67      0.40      0.50      1058

    accuracy                           0.91      9043
   macro avg       0.80      0.69      0.73      9043
weighted avg       0.90      0.91      0.90      9043

Recall: 0.4045368620037807
ROC-AUC: 0.9272396376476215

In [15]:
# Separate features and target
X3 = df3.drop(columns=['y'])
y3 = df3['y'].map({'no': 0, 'yes': 1})

# Train-test split (80:20)
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3,
    test_size=0.2,
    stratify=y3,
    random_state=42
)
print(X3_train.shape, X3_test.shape, y3_train.shape, y3_test.shape)

(32950, 20) (8238, 20) (32950,) (8238,)


In [16]:
num3_cols = X3_train.select_dtypes(include=np.number).columns.tolist()
cat3_cols = X3_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features in df3:", num3_cols)
print("Categorical features in df3:", cat3_cols)


Numerical features in df3: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical features in df3: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [17]:
# # Numerical preprocessing
# num_pipeline = Pipeline([
#     ('scaler', StandardScaler())
# ])

# # Categorical preprocessing
# cat_pipeline = Pipeline([
#     ('encoder', OneHotEncoder(handle_unknown='ignore'))
# ])

# Combine both preprocessors
preprocessor3 = ColumnTransformer([
    ('num', num_pipeline, num3_cols),
    ('cat', cat_pipeline, cat3_cols)
])
# separate pipeline for lr due to its need of nuemeric feature scaling
lr_pipeline3 = Pipeline([
    ('preprocessor', preprocessor3),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train
lr_pipeline3.fit(X3_train, y3_train)

# Predictions
y3_pred = lr_pipeline3.predict(X3_test)
y3_prob = lr_pipeline3.predict_proba(X3_test)[:, 1]

In [18]:
print(f"\n{'='*50}")
print("Logistic Regression on df3")
print(f"{'='*50}")

print("\nConfusion Matrix:")
print(confusion_matrix(y3_test, y3_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y3_test, y3_pred, zero_division=0))

print("Recall:", recall_score(y3_test, y3_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y3_test, y3_prob))
print("Average Precision:", average_precision_score(y3_test, y3_prob))
df3_scores["Logistic Regression"]=[recall_score(y3_test, y3_pred, zero_division=0), roc_auc_score(y3_test, y3_prob), average_precision_score(y3_test, y3_prob)]



Logistic Regression on df3

Confusion Matrix:
[[7146  164]
 [ 523  405]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95      7310
           1       0.71      0.44      0.54       928

    accuracy                           0.92      8238
   macro avg       0.82      0.71      0.75      8238
weighted avg       0.91      0.92      0.91      8238

Recall: 0.4364224137931034
ROC-AUC: 0.9424372906740885
Average Precision: 0.6356004239349451


In [19]:

# Preprocessing: One-Hot Encoding only
preprocessor3= ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat3_cols)
    ],
    remainder='passthrough'
)

# # Define models
# models = {
#     'Decision Tree': DecisionTreeClassifier(random_state=42),
#     'Random Forest': RandomForestClassifier(random_state=42),
#     'XGBoost': XGBClassifier(
#         random_state=42,
#         eval_metric='logloss'
#     )
# }

# Train and evaluate models
fitted_models3 = {}

for name, model in models.items():

    clf3 = Pipeline([
        ('preprocessor', preprocessor3),
        ('classifier', model)
    ])

    # Train
    clf3.fit(X3_train, y3_train)

    # Predictions
    y_pred = clf3.predict(X3_test)
    y_prob = clf3.predict_proba(X3_test)[:, 1]

    # Evaluation
    print(f"\n{'='*50}")
    print(f"Model: {name}", "on df3")
    print(f"{'='*50}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y3_test, y_pred, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(y3_test, y_pred, zero_division=0))

    print("Recall:", recall_score(y3_test, y_pred, zero_division=0))
    print("ROC-AUC:", roc_auc_score(y3_test, y_prob))
    print("Average Precision:", average_precision_score(y3_test, y_prob))
    df3_scores[name]=[recall_score(y3_test, y_pred, zero_division=0), roc_auc_score(y3_test, y_prob), average_precision_score(y3_test, y_prob)]

    fitted_models3[name] = clf3


Model: Decision Tree on df3

Confusion Matrix:
[[6875  435]
 [ 433  495]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      7310
           1       0.53      0.53      0.53       928

    accuracy                           0.89      8238
   macro avg       0.74      0.74      0.74      8238
weighted avg       0.89      0.89      0.89      8238

Recall: 0.5334051724137931
ROC-AUC: 0.7369488242369923
Average Precision: 0.33647050595857775

Model: Random Forest on df3

Confusion Matrix:
[[7129  181]
 [ 488  440]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      7310
           1       0.71      0.47      0.57       928

    accuracy                           0.92      8238
   macro avg       0.82      0.72      0.76      8238
weighted avg       0.91      0.92      0.91      8238

Recall: 0.47413793103448276
ROC-AUC: 0.946538235883768

In [20]:
# print("df1_scores:", df1_scores)
# print("df3_scores:", df3_scores)
# in the order recall, roc-auc, avg. precision
metrics = ['Recall', 'ROC-AUC', 'Average Precision']

df1_results = pd.DataFrame.from_dict(
    df1_scores,
    orient='index',
    columns=metrics
)

df3_results = pd.DataFrame.from_dict(
    df3_scores,
    orient='index',
    columns=metrics
)

print("DF1 Results:")
display(df1_results.round(4))

print("DF3 Results:")
display(df3_results.round(4))

DF1 Results:


,Recall,ROC-AUC,Average Precision
Logistic Regression,0.3478,0.9056,0.5451
Decision Tree,0.4707,0.6989,0.2790
Random Forest,0.4045,0.9272,0.6165
XGBoost,0.4915,0.9300,0.6238


DF3 Results:


,Recall,ROC-AUC,Average Precision
Logistic Regression,0.4364,0.9424,0.6356
Decision Tree,0.5334,0.7369,0.3365
Random Forest,0.4741,0.9465,0.6747
XGBoost,0.5700,0.9506,0.6764


# baseline model comparision
Across the four baseline classifiers, the models trained on df3 (Bank Additional Full) achieved better test-set performance than their counterparts trained on df1 (Bank Full).
All four classifiers demonstrated improvements in Recall, ROC-AUC and Average Precision.

which metric should we optimise on?
